# HW 5

Your Name: 

Due: Wednesday December 3 @ 11:59 PM EST

Extra Credit Deadline: Monday December 1 @ 11:59 PM EST

### Submission Instructions
Submit this `ipynb` file to Gradescope (this can also be done via the assignment on Canvas).  To ensure that your submitted `ipynb` file represents your latest code, make sure to give a fresh `Kernel > Restart & Run All` just before uploading the `ipynb` file to gradescope. **In addition:**
- Make sure your name is entered above
- Make sure you comment your code effectively
- If problems are difficult for the TAs/Profs to grade, you will lose points

### Tips for success
- Start early
- Make use of Piazza (also accessible through Canvas)
- Make use of Office Hours
- Remember to use cells and headings to make the notebook easy to read (if a grader cannot find the answer to a problem, you will receive no points for it)
- Under no circumstances may one student view or share their ungraded homework or quiz with another student [(see also)](http://www.northeastern.edu/osccr/academic-integrity), though you are welcome to **talk about** (*not* show each other your answers to) the problems.

In [1]:
# you might use the below modules on this lab
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from collections import Counter
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))

## Part 1: Logistic Regression (75 points)

### Part 1.1 Perceptron with Sigmoid (25 points)
Complete the function `linear_perceptron_sigmoid()` below (including docstring) which takes as arguments:

- `X`: a 2d-array (including bias column of 1s) with columns equal to $x$ features
- `y`: a 1d-array of labels (-1 or 1)
- `w`: an initial w vector of same dimension as the columns of `X`
- `alpha`: the learning rate, with default value of 1
- `max_iter`: the maximum number of iterations for the algorithm to run, with default value of `None`
    
The function should return only `w`, the final weight vector of the perceptron algorithm.  **YOU MUST IGNORE THE `pass` statement from the function when you're done**.

**Also** make sure the assert statement doesn't complain about the test case in the last cell of this part before moving on.

**There are some missing things in the following two block of codes** make sure you add those!!

In [2]:
def linear_perceptron_sigmoid(X, y, w, alpha = 1, max_iter = None):
    """
    Perceptron algorithm with sigmoid activation function.
    
    Args:
        X: 2d-array with bias column of 1s and feature columns
        y: 1d-array of labels (0 or 1)
        w: initial weight vector
        alpha: learning rate (default 1)
        max_iter: maximum iterations (default None)
    
    Returns:
        w: final weight vector
    """
    runalg = True
    i = 0
    iter = 0

    while runalg:
        # Calculate sigmoid prediction: σ(w^T x_i)
        z = X[i] @ w
        sigmoid_pred = 1 / (1 + np.exp(-z))
        
        # Update weights using gradient descent
        w = w + alpha * (y[i] - sigmoid_pred) * X[i]
        
        i += 1
        
        # Reset i and increment iter when we've gone through all observations
        if i >= X.shape[0]:
            i = 0
            iter += 1
            
            # Check max_iter condition
            if max_iter is not None and iter >= max_iter:
                print(f'w: {w}, iter: {iter}')
                runalg = False
                break
    
    return w

In [3]:
x1 = np.array([0.5, 2.5, 1, 4.0])
x2 = np.array([1.5, 2.5, 2, 3.5])
yi = np.array([0, 1, 0, 1])
w=np.array([0,-2,1])

# Create X
X = np.column_stack([np.ones(len(x1)), x1, x2])

myrun = linear_perceptron_sigmoid(X, yi, w, alpha = .04, max_iter = 1000)

expected_result = np.array([-4.03870189,4.82015984,-1.89457192])

assert np.allclose(myrun,expected_result), 'linear_perceptron_sigmoid() error'

w: [-4.03870189  4.82015984 -1.89457192], iter: 1000


### Part 1.2 Prediction using Logistic Regression (25 points)

Using the weights that you obtained in the previous block. Complete the prediction function and make sure the assert statement passes. 

**Hint** You may need to change the `yhati` to the desired form!


In [4]:
def predict_perceptron_sigmoid(x, w, pred0 = True):
    """
    Predict using sigmoid perceptron.
    
    Args:
        x: feature vector (with bias)
        w: weight vector
        pred0: if True, return 0/1; if False, return probability
    
    Returns:
        yhat: prediction (0/1 or probability)
    """
    z = x @ w
    sigmoid_pred = 1 / (1 + np.exp(-z))
    
    if pred0:
        yhat = 1 if sigmoid_pred >= 0.5 else 0
    else:
        yhat = sigmoid_pred
    
    return yhat

x1 = np.array([-8, 1, -.2, .4])
x2 = np.array([1.7, -.7, -.3, -1.3])

# Create X 
X = np.column_stack([np.ones(len(x1)), x1, x2])

yhati = np.apply_along_axis(predict_perceptron_sigmoid, 1, X, pred0 = False, w=myrun)
yhati = (yhati >= 0.5).astype(int)

expected_result = np.array([0,1,0,1])
assert np.allclose(yhati,expected_result),'predict_perceptron_sigmoid() error'

## Part 1.3 Real Data Logistic Regression (25 points)

Using the funtions you wrote in parts 1.1 and 1.2, and the data from Lab 4 (Read in for you below) apply `linear_perceptron_sigmoid`. Set up the numpy arrays for your `X` and `y` features to predict whether a song is in Major or Minor mode. Use the default `alpha` and set `max_iter=1000`. After fitting the model:

- Calculate the accuracy of the model
  - Use your `predict_perceptron_sigmoid` funtion from Part 1.2.


In [5]:
url = 'https://raw.githubusercontent.com/eaegerber/data/main/ds3000_spotify_scaled.csv'
df_spot_raw = pd.read_csv(url)
df_spot = df_spot_raw[["energy", "key", "loudness", "mode", "song_title", "artist_name"]]
df_spot.insert(loc=0, column='bias', value=1)
scale_columns = ["energy", "key", "loudness"]
for feat in scale_columns:
    df_spot[feat] = (df_spot[feat] - df_spot[feat].mean()) / df_spot[feat].std()

df_spot.head()

,bias,energy,key,loudness,mode,song_title,artist_name
0,1,-0.326724,0.452859,0.669783,0,DEEP IN THE WATER,Don Toliver
1,1,0.521290,1.573185,0.527116,0,Halo,Beyoncé
2,1,0.885408,1.013022,1.284685,1,OBVIOUS,Fordo
3,1,-1.299304,1.013022,-1.385809,0,Stairway to Heaven - Remaster,Led Zeppelin
4,1,0.209872,-0.947548,0.394105,1,ocd,ericdoa


In [6]:
# Set up X and y
X = df_spot[['bias', 'energy', 'key', 'loudness']].values
y = ((df_spot['mode'].values + 1) / 2).astype(int)  # Convert -1/1 to 0/1

# Initialize weights
w_init = np.zeros(X.shape[1])

# Train the model
w_final = linear_perceptron_sigmoid(X, y, w_init, alpha=1, max_iter=1000)

# Make predictions
y_pred = np.apply_along_axis(predict_perceptron_sigmoid, 1, X, pred0=True, w=w_final)

# Calculate accuracy
accuracy = np.mean(y_pred == y)
print(f'Accuracy: {accuracy:.4f}')


w: [ 2.89578022  1.51667504 -1.56310786  3.16165889], iter: 1000
Accuracy: 0.5455


## Part 2: Unsupervised Learning (25 points)

Build a simple **Cosine Similarity** ranking using the `europe_pop_data.zip` data to determine which countries in Europe is most similar to **Germany** based on the demographic features provided therein. Below, I've read in the data and scaled it for you (scaling is very important!).

Then, write a loop to compare **Germany** to all the other countries and print the sorted ranking list from the head and the tail.

In [7]:
# read in the data, clean it, add population density, discard categorical features, and scale again
df_EUpop = pd.read_csv('europe_pop_data.zip')
df_EUpop['area'] = df_EUpop['area'].str.replace(r'\D', '', regex=True).astype(int)
df_EUpop['population'] = df_EUpop['population'].str.replace(r'\D', '', regex=True).astype(int)
df_EUpop['pop_density'] = (df_EUpop.population/df_EUpop.area).round(1)
col_num_list = ['male_life_expectancy', 'female_life_expectancy', 'birth_rate', 'death_rate', 'pop_density']
df_EUpop_num = df_EUpop.loc[:, col_num_list]
# by subtracting each feature by the mean and dividing by the standard deviation, outputs will be "unit invariant"
df_EUpop_num_scaled = pd.DataFrame()
for feat in df_EUpop_num.columns:
    df_EUpop_num_scaled[f'{feat}_scaled'] = ((df_EUpop_num[feat] - df_EUpop_num[feat].mean()) / df_EUpop_num[feat].std()).round(3)

df_EUpop_num_scaled.head(3)

,male_life_expectancy_scaled,female_life_expectancy_scaled,birth_rate_scaled,death_rate_scaled,pop_density_scaled
0,0.626,0.608,-0.198,-0.356,-0.187
1,0.562,0.434,0.253,-0.120,0.827
2,0.689,1.197,1.154,-0.490,-0.126


In [8]:
germany = df_EUpop_num_scaled.iloc[3].to_numpy()
germany

array([ 0.562,  0.538, -0.288,  0.183,  0.284])

In [9]:
# this creates empty lists to fill in with the dot products and cos(theta) of each country relative to Germany
bel_dot_products = []
bel_cosines = []
country_names = []

# this goes iteratively (loops) through each country in the data set and:
# (a) calculates the dot product between Germany and the country
# (b) calculates the cosine(theta) between Germany and the country
for country in range(df_EUpop_num_scaled.shape[0]):
    country_vec = df_EUpop_num_scaled.iloc[country].to_numpy()
    dot_product = np.dot(germany, country_vec)
    cosine = dot_product / (np.linalg.norm(germany) * np.linalg.norm(country_vec))
    
    bel_dot_products.append(dot_product)
    bel_cosines.append(cosine)
    country_names.append(df_EUpop.iloc[country]['country_name'])

# Create dataframe and sort by cosine similarity
similarity_df = pd.DataFrame({
    'country': country_names,
    'dot_product': bel_dot_products,
    'cosine_similarity': bel_cosines
}).sort_values('cosine_similarity', ascending=False)

print("Top 5 most similar countries to Germany:")
print(similarity_df.head())
print("\nTop 5 least similar countries to Germany:")
print(similarity_df.tail())

Top 5 most similar countries to Germany:
     country  dot_product  cosine_similarity
3    Germany     0.802377           1.000000
34  Slovenia     0.641743           0.752826
32  Portugal     1.025283           0.722466
0    Austria     0.617684           0.702891
1    Belgium     0.689380           0.683849

Top 5 least similar countries to Germany:
       country  dot_product  cosine_similarity
8      Belarus    -1.180492          -0.729809
31  Montenegro    -1.453518          -0.790653
15      Russia    -2.143975          -0.793683
12     Moldova    -2.119100          -0.901754
16    Slovakia    -0.845589          -0.909317
